- With the **DataFrame** API, there are only two modes for table interaction 'overwite' and 'append'.  Also, it does not allow individual record manipulation or CRUD actions into a single operation. In contrast the DetlaTable API is more robust
- Docs:  https://docs.delta.io/api/latest/python/spark/

In [0]:
from delta.tables import DeltaTable

In [0]:
"""
first, read in as dataframe and display data
"""
df = spark.read.table('workspace.pyspark_learning.country_regions')
display(df)

In [0]:
"""
read in country_regions table not as a dataframe but as a delta table
"""
dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_regions')
type(dt)

In [0]:
"""
read parquet file as a delta table
"""
dt2 = DeltaTable.forPath(spark, '/Volumes/workspace/pyspark_learning/raw_files/pyspark/countries_dataset/detaLake/')
type(dt2)

Record Munipulation - CRUD
- delta table api interacts with the delta tables in the catalog directly.  It needs no dataframe as a 'middle-man'.
- Whereas the delta table API interacts with individual table records, if using a dataframe, one must edit the dataframe the 'overwrite' a table to remove or update records

Delete Records

In [0]:
"""
Delete record
NOTE:  This does not delete a record locally, like in a dataframe, but 'dt' is a reference to the table itself.  What I do on 'dt' directly affects the table
"""
dt.delete("name = 'America'")

In [0]:
"""
Display 'df', and the record with America is gone.  Compare to cell #2 above
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()


Update Records

In [0]:
"""
Update records
"""

from pyspark.sql.functions import lit, upper


In [0]:
"""
for records where in column, 'name' the value is 'Asia', perform the following:
steps: 1. identify the condition
       2. set the column values
Note: set uses a dictionary
"""
dt.update(
    condition = "name = 'Asia'",
    set = {
        "id": lit(100),
        "name": upper("name")
    }
)

In [0]:
"""
verify update
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()


In [0]:
"""
To update all records, set condition = None
Below I am setting all name values to uppercase
"""
dt.update(
    condition = None,
    set = {
        "name": upper("name")
    }
)

In [0]:
"""
verify update
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

Merge Records
- upsert using 'merge'

In [0]:
from delta.tables import *

In [0]:
"""
import table to delta object
"""
dt = DeltaTable.forName(spark, 'workspace.pyspark_learning.country_regions')
type(dt)

In [0]:
"""
a delta object cannot be displayed so it needs to be converted to a dataframe for display purposes
NOTE:  chained methods
"""
dt.toDF().display()

In [0]:
"""
delete the ASIA and EUROPE records
"""
dt.delete("id = 100")

In [0]:
dt.delete("id = 20")

In [0]:
"""
verify
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

In [0]:
"""
read original regions csv data into dataframe
ddl schema is used here for simplicity
"""
schema = "id int, name string"
regions_df = spark.read.csv('/Volumes/workspace/pyspark_learning/raw_files/pyspark/countries_dataset/csv_data/country_regions/country_regions.csv', header=True, schema=schema)
display(regions_df)
regions_df.printSchema()

In [0]:
"""
use regions_df to upsert into the country_regions table, inserting and updating records
- When ID's match update from dataframe to table
- When ID's do not match insert from dataframe into table

Target is on the left, source is on the right of each statement
dt is the table, aliased to 'target'
df is is the dataframe, aliased to 'source

"""

dt.alias("target").\
    merge(
        regions_df.alias("source"),
        "target.id = source.id"
    ).\
    whenMatchedUpdate(
        set = {
            "target.name": "source.name",
        }
    ).\
    whenNotMatchedInsert(
        values = {
            "target.id" : "source.id",
            "target.name": "source.name"
        }
    ).execute()
            

In [0]:
"""
Verify table update
"""
spark.read.table('workspace.pyspark_learning.country_regions').display()

Transaction log

In [0]:
"""
Create data for transaction log
NOTE:  the write method has no '.delta' method; therefore CANNOT do:  df.write.delta.  Must use format method with chained save method.
"""
data = [
  {"id": 1, "name": "Alice", "score": 85},
  {"id": 2, "name": "Bob", "score": 90},
  {"id": 3, "name": "Charlie", "score": 70},
  {"id": 4, "name": "Diana", "score": 92}, 
  {"id": 5, "name": "Ethan", "score": 88},
  {"id": 6, "name": "Fiona", "score": 81},
  {"id": 7, "name": "George", "score": 74},
  {"id": 8, "name": "Hannah", "score": 95},
  {"id": 9, "name": "Ian", "score": 69},
  {"id": 10, "name": "Jasmine", "score": 87},

]

df = spark.createDataFrame(data)
df.write.format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data', mode='append')



In [0]:
"""
add more students to enhance log
"""

data2 = [
  {"id": 11, "name": "Kevin", "score": 83},
  {"id": 12, "name": "Lena", "score": 91},
  {"id": 13, "name": "Marcus", "score": 77},
  {"id": 14, "name": "Nina", "score": 89}, 
  {"id": 15, "name": "Oscar", "score": 72},
  {"id": 16, "name": "Priya", "score": 94},
  {"id": 17, "name": "Quinn", "score": 79},
  {"id": 18, "name": "Ravi", "score": 88},
  {"id": 19, "name": "Sophie", "score": 86},
  {"id": 20, "name": "Tom", "score": 80},

]

df2 = spark.createDataFrame(data2)
df2.write.format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data', mode='append')

In [0]:
"""
Delete all scores less than 80
"""
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, '/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data')
dt.delete("score < 80")

In [0]:
"""
create more logs
"""

for i in range(50):
    spark.createDataFrame(data2).write.mode('append').format('delta').save('/Volumes/workspace/pyspark_learning/raw_files/pyspark/output_files/delta_lake/student_data')